# Part II: Green Energy Planning

## Problem Overview

Rhode Island (RI) passed Act on Climate legislation in 2021, which established legally binding reduction targets on greenhouse gas emissions for the state, with the goal of moving RI towards a cleaner energy system. This problem is concerned with optimizing investment in green energy sources based on RI's historical electricity demand.

Rhode Island needs electricity throughout the day, but solar and wind energy generation can change with the weather. Solar panels generate less electricity at night, and wind turbines generate less electricity when the wind is weak. **How much solar and wind capacity should the state invest in, and how much battery storage should it purchase, to meet its energy demand at the lowest possible cost?**

## Load necessary imports

In [ ]:
# DO NOT EDIT
import sys
from pathlib import Path
import cvxpy as cvx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if sys.version_info[:3] != (3, 10, 6):
    raise RuntimeError("Select your Python 3.10.6 CS410 environment for the kernel.")
candidates = [Path("."), Path(".."), Path("brown-cs410-f26-constrained-optimization-assignment-5")]
DATA_DIR = next((path for path in candidates if (path / "RI_wind.csv").is_file()), None)

## The Data

The data set comprises 8,760 aligned **hourly** observations in Rhode Island, each with the following features:

| Name | Meaning |
| :--- | :--- |
| `solar_cf` | **Solar capacity factor** (output per unit of installed solar capacity) |
| `wind_cf` | **Wind capacity factor** (output per unit of installed wind capacity) |
| `demand_MW` | Electricity demand in megawatts (MW) |

In this data set, solar and wind capacity is reported in megawatts (MW), which is a measure of the *rate* of generation at any given moment. In contrast, energy generated or stored over time is measured in megawatt-hours (MWh). For example, a 1 MW solar installation operating at a capacity factor of 0.75 for 10 hours generates 7.5 MWh.

Also given are the costs of the various system components:

| Name | Meaning |
| :--- | :--- |
| `solar_cost` | Installation cost per MW of solar capacity (thousand dollars/MW) |
| `wind_cost` | Installation cost per MW of wind capacity (thousand dollars/MW) |
| `battery_cost` | Installation cost per MWh of battery capacity (thousand dollars/MWh) |

Finally, electricity cannot be stored like inventory in a warehouse. It dissipates. In this homework, we assume an **hourly retention rate $r$**, which is the fraction of energy stored in a battery that persists from one hour to the next. For example, if $r=0.9$, then only 9 of 10 MWh of energy stored during the previous hour would remain.

| Name | Meaning |
| :--- | :--- |
| `battery_retention` | Fraction of battery energy retained each hour |

Run the cell below to preview the data.

In [ ]:
# DO NOT EDIT
wind_data = pd.read_csv(DATA_DIR / "RI_wind.csv")
solar_data = pd.read_csv(DATA_DIR / "RI_solar.csv")
demand_data = pd.read_csv(DATA_DIR / "RI_demand.csv")

# Use these args in your CVXPY model!!
args = {
    "solar_cf": solar_data["s_cfs"].to_numpy(dtype=float),
    "wind_cf": wind_data["w_cfs"].to_numpy(dtype=float),
    "demand": demand_data["demand (MW)"].to_numpy(dtype=float),
    # Costs are in thousands of dollars per MW or MWh.
    "solar_cost": 14.772123,
    "wind_cost": 15.913707,
    "battery_cost": 7.351353,
    "battery_retention": 0.995,
}
target_length = len(args["demand"])
print(f"Loaded {target_length:,} aligned hourly observations.")
print("Displaying the first five rows:")
display(pd.DataFrame({
    "solar_cf": args["solar_cf"],
    "wind_cf": args["wind_cf"],
    "demand_MW": args["demand"],
}).head())

## Formulating the Constrained Optimization Problem

To formulate an optimization problem as a mathematical program, beyond the data, there are three more essential pieces:

1. **Decision variables:** the optimizer's choices
2. **Objective:** what the optimizer should minimize
3. **Constraints:** what every feasible solution must satisfy

Before reading on, think about how to define these three components for green energy planning. **Spend some time brainstorming about the decision variables, in particular, and recall from lecture the need for implicit state variables in dynamic planning problems.**

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Decision Variables

The optimizer decides how much wind and solar capacity and battery storage capacity to purchase. 

Additionally, the optimizer must make some "implicit decisions," namely how much energy to store in the battery during each hour.

</div>

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Objective

Minimize the total installation cost of the purchased solar, wind, and battery storage capacities.

Because costs are measured in thousands of dollars per MW or MWh, the objective value is measured in thousands of dollars.

</div>

<div style="background-color: rgba(30, 144, 255, 0.15); padding: 15px; border-radius: 8px;">

### Constraints

Your energy plan must meet the following constraints:
- Meet the demand at each hour.
- Electricity generated in excess of demand can be stored for later use or left unused.
- The battery starts and ends at 50% of its storage capacity.
- Never store more energy than the purchased battery storage capacity.
- All decision variables are nonnegative, i.e. installed capacities and stored energy can't be negative.

</div>

## Tasks

At this point, you should understand how the data, decision variables, objective, and constraints fit together. Your first task is to complete the mathematical formulation of the problem.

### Task 2.1: Mathematical Formulation

Write the mathematical formulation of the green energy planning problem. Identify the constants, decision variables, objective function, and constraints. You can write in LaTeX (and submit as a pdf) or on paper by hand (and submit as an image). Please make sure to state the file name in `README.md`.


### Task 2.2: Build and Solve the Model with `CVXPY`

Complete `solve_energy_plan(model_args)` by translating your mathematical formulation into `CVXPY`. We encourage you to use an LLM for assistance with this task.

Your function should return the optimized solar, wind, and battery storage capacities and the minimum total cost using the format described in its docstring.

We recommend using the CLARABEL solver (i.e., `model.solve(solver="CLARABEL")`) to solve your `CVXPY` problem formulation.


In [ ]:
def solve_energy_plan(model_args):
    """
    Build and solve an energy-planning model for the supplied data.
    
    Input: model_args is a dictionary containing the following keys:
        solar_cf: array of solar capacity factors (length T)
        wind_cf: array of wind capacity factors (length T)
        demand: array of demand values (length T)
        solar_cost: cost of solar capacity (in thousands of dollars per MW)
        wind_cost: cost of wind capacity (in thousands of dollars per MW)
        battery_cost: cost of battery capacity (in thousands of dollars per MWh)
        battery_retention: fraction of energy retained in the battery each hour (between 0 and 1)
    
    Output: a dictionary containing the following keys:
        solar_capacity: optimal solar capacity (MW)
        wind_capacity: optimal wind capacity (MW)
        battery_capacity: optimal battery capacity (MWh)
        total_cost: total cost of the optimal plan (in thousands of dollars)
    """
    
    # We unpacked the model_args for you for convenience.
    solar_cf = model_args["solar_cf"]
    wind_cf = model_args["wind_cf"]
    demand = model_args["demand"]
    solar_cost = model_args["solar_cost"]
    wind_cost = model_args["wind_cost"]
    battery_cost = model_args["battery_cost"]
    battery_retention = model_args["battery_retention"]
    
    # TODO: Build and solve the CVXPY model, then return the requested values.
    pass

#### Provided Green Energy Planning Model Tests

While developing your model, you can run these tests to **Verify** and then **Refine** the LLM-generated or your own code.

We built these tests using synthetic data and on a small fraction (one week / 168 hours) of the RI dataset.

You can add to this testbed, but we do not recommend editing our tests!!

In [ ]:
# DO NOT EDIT
def check_expected_plan(model_args, expected):
    """Solve one case and compare the returned purchase decisions and cost."""
    result = solve_energy_plan(model_args)
    actual = [
        result["solar_capacity"],
        result["wind_capacity"],
        result["battery_capacity"],
        result["total_cost"],
    ]
    np.testing.assert_allclose(actual, expected, atol=1e-3, rtol=0)


def test_generation_cases():
    """Check three small cases that do not require battery storage."""
    cases = [
        ({
            "wind_cf": [1, 1, 0, 0], "solar_cf": [0, 0, 1, 1],
            "solar_cost": 1, "wind_cost": 2, "battery_cost": 1000,
            "demand": [5, 5, 5, 5], "battery_retention": 1.0,
        }, [5, 5, 0, 15]),
        ({
            "wind_cf": [1, 1, 1, 1], "solar_cf": [0, 0, 1, 1],
            "solar_cost": 1, "wind_cost": 2, "battery_cost": 1000,
            "demand": [4, 4, 4, 4], "battery_retention": 1.0,
        }, [0, 4, 0, 8]),
        ({
            "wind_cf": [1, 1, 0.5, 0.5], "solar_cf": [0, 0, 1, 1],
            "solar_cost": 1, "wind_cost": 2, "battery_cost": 1000,
            "demand": [4, 4, 4, 4], "battery_retention": 1.0,
        }, [2, 4, 0, 10]),
    ]
    for model_args, expected in cases:
        check_expected_plan(model_args, expected)


def test_battery_case():
    """Check a small case in which purchasing battery capacity is required."""
    model_args = {
        "wind_cf": [1, 1, 0.5, 0], "solar_cf": [0, 0, 1, 0],
        "solar_cost": 1, "wind_cost": 2, "battery_cost": 10,
        "demand": [4, 4, 4, 4], "battery_retention": 1.0,
    }
    check_expected_plan(model_args, [1, 6, 8, 93])


# ONE WEEK OF RI DATA (168 HOURS)
development_hours = 168
development_args = {
    **args,
    "solar_cf": args["solar_cf"][:development_hours],
    "wind_cf": args["wind_cf"][:development_hours],
    "demand": args["demand"][:development_hours],
}
development_results = solve_energy_plan(development_args)
development_results


def test_development_plan():
    """Check the purchase decisions and cost for the one-week dataset."""
    actual = [
        development_results["solar_capacity"],
        development_results["wind_capacity"],
        development_results["battery_capacity"],
        development_results["total_cost"],
    ]
    expected = [1760.958237, 3183.034885, 7049.701410, 128491.819809]
    np.testing.assert_allclose(actual, expected, atol=1e-2, rtol=1e-6)


test_generation_cases()
test_battery_case()
test_development_plan()
print("All tests passed.")

#### Testing on the Full Rhode Island Dataset

Once your model works on the tests provided, you are ready to run it on the full dataset of 8,760 hours. 

This solve may take around 20 or 30 seconds (or longer) depending on your computer's specifications.

In [ ]:
# DO NOT EDIT
results = solve_energy_plan(args)

# Display the optimized values from `results`
print("Optimal values returned by solve_energy_plan:")
display(pd.Series({
    key: results[key]
    for key in ("solar_capacity", "wind_capacity", "battery_capacity", "total_cost")
}, name="Optimal value").to_frame())

#### Full-Year Answer Check

After the full-year solve completes, run the cell below to check its purchased capacities and minimum cost against the expected result.

In [ ]:
# DO NOT EDIT
def test_rhode_island_plan():
    """Check the purchased capacities and minimum cost for the full dataset."""
    actual = [
        results["solar_capacity"],
        results["wind_capacity"],
        results["battery_capacity"],
        results["total_cost"],
    ]
    expected = [4005.756491, 3164.350201, 21822.710916, 269956.520906]
    np.testing.assert_allclose(actual, expected, atol=1e-2, rtol=1e-6)


test_rhode_island_plan()
print("Full-year Rhode Island answer check passed.")

### Task 2.3: Does More Battery Capacity Help?

The original model does not place an upper bound on the battery capacity Rhode Island may purchase. We will now investigate how a battery-purchase limit affects the optimal energy plan.

Your final task is to complete `solve_energy_plan_with_limit(model_args, battery_limit)` by (slightly) adapting your Task 2.2 implementation. The supplied `battery_limit` argument is the maximum battery storage capacity the model may purchase, in MWh. 

The provided loop will solve your model with various values of the battery limit parameter, and record the resulting capacities and total costs.

*This experiment may take 2-3 minutes to run.*


In [ ]:
def solve_energy_plan_with_limit(model_args, battery_limit):
    """
    Build and solve the energy-planning model with a battery-purchase limit.

    Inputs:
        model_args: energy-planning data and costs, as described in
            solve_energy_plan()
        battery_limit: maximum battery capacity that may be purchased (MWh)

    Output: a dictionary containing the following keys:
        solar_capacity: optimal solar capacity (MW)
        wind_capacity: optimal wind capacity (MW)
        battery_capacity: optimal battery capacity (MWh)
        total_cost: total cost of the optimal plan (in thousands of dollars)
    """
    # TODO: Copy and adapt your Task 2.2 implementation so that the amount of battery capacity purchased respects battery_limit.
    pass


# DO NOT EDIT: Run the battery-limit experiment and record its results.
battery_limits = np.array([10_000, 13_000, 16_000, 20_000, 22_000, 40_000])  # MWh
rows = []

for limit in battery_limits:
    result = solve_energy_plan_with_limit(args, float(limit))
    rows.append({
        "battery_limit": float(limit),
        **result,
    })

#### Visualizations

After solving the model for a set of battery limits, run the provided visualization cell below. This cell will generate:

1. Installed solar/wind (MW) capacities versus `B`. 
2. Installed battery (MWh) versus `B`. 
3. Minimum total installation cost, in thousands of dollars, versus `B`.

In [ ]:
# DO NOT EDIT

capacity_table = pd.DataFrame(rows)
display(capacity_table)
costs = capacity_table["total_cost"].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for key in ("solar", "wind"):
    axes[0].plot(battery_limits, capacity_table[key + "_capacity"], marker="o", label=key)
axes[0].set_ylabel("Installed generation (MW)")
axes[0].legend()
axes[1].plot(battery_limits, capacity_table["battery_capacity"], marker="o", label="Installed battery")
axes[1].plot(battery_limits, battery_limits, "--", color="gray", label="Purchase limit")
axes[1].set_ylabel("Installed storage (MWh)")
axes[1].set_xlabel("Battery purchase limit B (MWh)")
axes[1].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(battery_limits, costs, marker="o")
plt.axhline(results["total_cost"], linestyle="--", color="gray", label="Unrestricted optimum")
plt.xlabel("Battery purchase limit B (MWh)")
plt.ylabel("Minimum total installation cost (thousand dollars)")
plt.legend()
plt.tight_layout()
plt.show()

### MCQs

In your `README.md`, answer the following MCQs:

#### Task 2.3.1
**Why does allowing more battery capacity generally reduce the total installed solar and wind capacity?**  

A) Higher battery capacities increase the solar and wind capacity factors.  
B) More energy can be stored during periods of high generation and used during periods of lower generation.  
C) Increasing the battery limit reduces the hourly electricity demand that the energy system must meet.  
D) Batteries create additional electricity when solar and wind generation are low.  

#### Task 2.3.2
**When the battery purchase limit reaches 40,000 MWh, what reason best explains why the optimizer installs only about 21,823 MWh?**

A) The solver cannot represent battery capacities more than 22,000 MWh.  
B) The battery must remain half full, so only half the limit may be purchased.  
C) Beyond this point, additional battery capacity costs more than it saves elsewhere in the system.  
D) Battery capacity greater than 21,823 MWh would exceed electricity demand. 

#### Task 2.3.3
**Why doesn't the minimum total installation cost increase even when the maximum battery capacity increases?**

A) Increasing the limit reduces the cost of battery storage.  
B) Plans allowed under the smaller limit remain feasible as additional plans become feasible.  
C) The law requires Rhode Island to purchase the maximum available capacity.  
D) Increasing the limit reduces electricity demand.
